# L13 - Il neurone artificiale e la separazione lineare

Eseguire le celle in ordine con `Maiusc + Invio`.

In [ ]:
# Funzione di controllo degli esercizi: eseguire questa cella, non modificarla.
def controlla(numero, prova):
    try:
        esito = bool(prova())
    except Exception as e:
        print(f"Esercizio {numero}: da rivedere ({type(e).__name__})")
        return
    print(f"Esercizio {numero}: " + ("corretto" if esito else "da rivedere"))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# I quattro punti della tabella di verità, uno per riga
P = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

## 1. Il neurone come funzione

Uscita del neurone a gradino: 1 se $w \cdot x + b \geq 0$, altrimenti 0. Con la matrice `P` si calcolano le uscite dei quattro punti in una sola espressione.

In [ ]:
def neurone(X, w, b):
    """Uscite (0 o 1) del neurone a gradino per tutte le righe di X."""
    return np.where(X @ w + b >= 0, 1, 0)

print("AND:", neurone(P, np.array([1, 1]), -1.5))
print("OR: ", neurone(P, np.array([1, 1]), -0.5))

## 2. La retta di separazione

Il neurone divide il piano con la retta $w_1 x_1 + w_2 x_2 + b = 0$. La funzione `disegna` mostra i punti (cerchi per uscita 0, quadrati per uscita 1) e la retta del neurone.

In [ ]:
def disegna(X, y, w, b, titolo=""):
    """Punti di X con etichette y (0 o 1) e retta del neurone con pesi w e bias b."""
    plt.scatter(X[y == 0, 0], X[y == 0, 1], marker="o", s=90, label="uscita 0")
    plt.scatter(X[y == 1, 0], X[y == 1, 1], marker="s", s=90, label="uscita 1")
    x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 50)
    if w[1] != 0:
        plt.plot(x1, -(w[0] * x1 + b) / w[1], color="black", label="retta del neurone")
    else:
        plt.axvline(-b / w[0], color="black", label="retta del neurone")
    plt.xlim(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
    plt.ylim(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
    plt.title(titolo)
    plt.legend()
    plt.show()

y_and = np.array([0, 0, 0, 1])
disegna(P, y_and, np.array([1, 1]), -1.5, "AND: w = (1, 1), b = -1.5")

## 3. Da che parte della retta

Il valore $z = w \cdot x + b$ è positivo da una parte della retta, negativo dall'altra, zero sulla retta. Il vettore dei pesi $w$ è perpendicolare alla retta e punta verso la parte in cui $z > 0$.

In [ ]:
w = np.array([1, 1])
b = -1.5
for x in P:
    print(x, "z =", x @ w + b)

## 4. XOR: una ricerca sistematica

XOR vale 1 quando esattamente uno degli ingressi vale 1. La cella prova tutte le combinazioni di pesi e bias tra -3 e 3 con passo 0.25 (più di 15 000 combinazioni) e conta quante calcolano XOR.

In [ ]:
y_xor = np.array([0, 1, 1, 0])
valori = np.arange(-3, 3.01, 0.25)
trovate = 0
provate = 0
for w1 in valori:
    for w2 in valori:
        for b in valori:
            provate += 1
            if np.array_equal(neurone(P, np.array([w1, w2]), b), y_xor):
                trovate += 1
print("combinazioni provate:", provate, " combinazioni che calcolano XOR:", trovate)

## Esercizi

**Esercizio 1 (base).** Trovare pesi `w_nand` e bias `b_nand` per NAND (uscita 0 solo per (1, 1)) e disegnare punti e retta con `disegna`.

In [ ]:
y_nand = np.array([1, 1, 1, 0])
w_nand = np.array([-1, -1])
b_nand = 1.5
disegna(P, y_nand, w_nand, b_nand, "NAND")

In [ ]:
controlla(1, lambda: np.array_equal(neurone(P, w_nand, b_nand), y_nand))

**Esercizio 2 (base).** Trovare `w_imp` e `b_imp` per la funzione "x1 AND NOT x2" (uscita 1 solo per (1, 0)).

In [ ]:
y_imp = np.array([0, 0, 1, 0])
w_imp = np.array([1, -1])
b_imp = -0.5
disegna(P, y_imp, w_imp, b_imp, "x1 AND NOT x2")

In [ ]:
controlla(2, lambda: np.array_equal(neurone(P, w_imp, b_imp), y_imp))

**Esercizio 3 (standard).** Il neurone con $w = (1, 1)$ e $b = -1.5$ calcola AND. Moltiplicando pesi e bias per lo stesso numero positivo la retta non cambia. Verificarlo: calcolare `uscite_x10`, le uscite del neurone con pesi e bias moltiplicati per 10, e `uscite_meno`, con pesi e bias moltiplicati per -1. Che cosa succede nel secondo caso? Rispondere in una cella di testo.

In [ ]:
uscite_x10 = neurone(P, 10 * np.array([1, 1]), 10 * -1.5)
uscite_meno = neurone(P, -1 * np.array([1, 1]), -1 * -1.5)
print(uscite_x10, uscite_meno)

In [ ]:
controlla(3, lambda: np.array_equal(uscite_x10, [0, 0, 0, 1]) and np.array_equal(uscite_meno, [1, 1, 1, 0]))

**Esercizio 4 (standard).** Punti a due dimensioni: altezza (in metri) e massa (in kg) di alcuni animali, etichetta 1 per i cani di taglia grande. Trovare `w_cani` e `b_cani` che separano correttamente tutti i punti e disegnarli.

In [ ]:
X_cani = np.array([[0.25, 5], [0.30, 8], [0.35, 10], [0.60, 30], [0.70, 38], [0.65, 45]])
y_cani = np.array([0, 0, 0, 1, 1, 1])
w_cani = np.array([0, 1])
b_cani = -20
disegna(X_cani, y_cani, w_cani, b_cani, "cani di taglia grande")

In [ ]:
controlla(4, lambda: np.array_equal(neurone(X_cani, w_cani, b_cani), y_cani))

**Esercizio 5 (approfondimento).** Ripetere la ricerca sistematica della sezione 4 per le 16 funzioni logiche di due ingressi (tutte le possibili liste di 4 uscite 0/1). Costruire `separabili`, la lista delle funzioni (come tuple di 4 valori) per cui esiste almeno una combinazione di pesi e bias. Quante sono? Quali mancano?

In [ ]:
separabili = []
for w1 in valori:
    for w2 in valori:
        for b in valori:
            u = tuple(int(v) for v in neurone(P, np.array([w1, w2]), b))
            if u not in separabili:
                separabili.append(u)
print(len(separabili))
print("mancano:", [f for f in [(a, c, d, e) for a in (0, 1) for c in (0, 1) for d in (0, 1) for e in (0, 1)] if f not in separabili])

In [ ]:
controlla(5, lambda: len(separabili) == 14 and (0, 1, 1, 0) not in separabili and (1, 0, 0, 1) not in separabili)